# Sample Model Training Scripts

This notebook documents the sample code scripts to preprocess the documents and to train the models. This seeks to provide an easy reference for users to customise and train their own models.

## Libraries

In [ ]:
import os
import pandas as pd
import shutil
import random
import io
import json
import pickle
import boto3
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import vstack
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import joblib
import time
from dotenv import load_dotenv
from botocore.config import Config
import numpy as np
import shap
import lime
from lime.lime_text import LimeTextExplainer

## Preprocessing

In [ ]:

def remove_stop_words(text: str) -> str:
        stop_words = set(stopwords.words('english'))
        words = text.split()
        return ' '.join(word for word in words if word not in stop_words)

def clean_text(text: str) -> str:
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def preprocess_pdf(self, file_content: bytes, filename: str) -> tuple[str, str]:
        try:
            pdf_stream = io.BytesIO(file_content)
            reader = PdfReader(pdf_stream)
            extracted_text = ""
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"
            cleaned_text = self.clean_text(extracted_text)
            final_text = self.remove_stop_words(cleaned_text)
            output_path = os.path.join(os.getcwd(), f"{filename}_extracted.txt")
            with open(output_path, "w", encoding="utf-8") as text_file:
                text_file.write(final_text)
            return output_path, final_text
        except Exception as e:
            print(f"Error processing PDF: {e}")
            return None, None
        


## Excerpt Extraction

In [ ]:
import random

def extract_intro_middle_conclusion(text, topic, max_tokens=20000):  
    words = text.split()
    total_words = len(words)

    # **Set dynamic extraction ratios based on document length**
    if total_words < 5000:
        intro_ratio, middle_ratio, conclusion_ratio = 0.15, 0.15, 0.15
    elif total_words < 20000:
        intro_ratio, middle_ratio, conclusion_ratio = 0.08, 0.12, 0.08
    elif total_words < 50000:
        intro_ratio, middle_ratio, conclusion_ratio = 0.04, 0.08, 0.04
    else:
        intro_ratio, middle_ratio, conclusion_ratio = 0.02, 0.06, 0.02

    # **Ensure at least 100 words extracted**
    min_length = 100
    intro_end = max(int(total_words * intro_ratio), min_length)
    conclusion_start = max(int(total_words * (1 - conclusion_ratio)), total_words - min_length)

    # **Middle Section Sampling**
    middle_start = intro_end
    middle_end = conclusion_start
    middle_range = words[middle_start:middle_end]

    if len(middle_range) > min_length:
        middle_sample_size = int(len(middle_range) * middle_ratio)
        middle_sample = random.sample(middle_range, min(middle_sample_size, len(middle_range)))
    else:
        middle_sample = middle_range

    # **Assemble Hybrid Text**
    intro_text = words[:intro_end]
    middle_text = middle_sample
    conclusion_text = words[conclusion_start:]
    
    hybrid_text_words = intro_text + middle_text + conclusion_text

    # **Estimate Token Usage**
    estimated_tokens = len(hybrid_text_words) * 1.3  

    # **Truncate if Exceeding Token Limit**
    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)  
        hybrid_text_words = hybrid_text_words[:allowed_words]

        # **Redistribute to keep sections separate but within limit**
        new_intro_end = max(int(len(hybrid_text_words) * (intro_ratio / (intro_ratio + middle_ratio + conclusion_ratio))), min_length)
        new_conclusion_start = max(len(hybrid_text_words) - int(len(hybrid_text_words) * (conclusion_ratio / (intro_ratio + middle_ratio + conclusion_ratio))), min_length)

        intro_text = hybrid_text_words[:new_intro_end]
        middle_text = hybrid_text_words[new_intro_end:new_conclusion_start]
        conclusion_text = hybrid_text_words[new_conclusion_start:]

    return " ".join(intro_text), " ".join(middle_text), " ".join(conclusion_text)


## Random Forest Model

The code assumes that the extracted sample document excerpts (cleaned excerpts that are extracted from entire document) are already in the directory "Extracted_Sample_Data".

In [ ]:
import os
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTEENN
from nltk.tokenize import word_tokenize
import joblib


# Load & Process Documents
print("Processing Documents...")
data_dir = "Extracted_Sample_Data"
documents, labels, filenames = [], [], []

for main_topic in os.listdir(data_dir):
    main_topic_path = os.path.join(data_dir, main_topic)
    if os.path.isdir(main_topic_path):  
        for root, _, files in os.walk(main_topic_path):  
            sub_topic = root.replace(data_dir + "/", "").strip()
            sub_topic_cleaned = sub_topic.split("/")[-1]  
            for file in files:
                if file.endswith(".txt"):
                    file_path = os.path.join(root, file)
                    with open(file_path, "r", encoding="utf-8") as f:
                        content = f.read()
                    documents.append(content)
                    labels.append(sub_topic_cleaned)
                    filenames.append(file)

# Convert to DataFrame
df = pd.DataFrame({"Filename": filenames, "Text": documents, "Label": labels})
print(f"Loaded {len(df)} documents for classification.")

# Extract TF-IDF Features
print("Extracting TF-IDF Features...")
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    max_df=0.85,
    min_df=2,
    ngram_range=(1, 3)
)
X = tfidf_vectorizer.fit_transform(df["Text"])
y = df["Label"]

# Apply Combined Resampling Strategy
print("Applying combined resampling strategy...")
smote = SMOTE(random_state=42, k_neighbors=1)
smote_enn = SMOTEENN(smote=smote, random_state=42)
X_resampled, y_resampled = smote_enn.fit_resample(X, y)

# Apply Recursive Feature Elimination (RFE)
print("Applying Recursive Feature Elimination (RFE)...")
rf_model = RandomForestClassifier(
    n_estimators=5000,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rfe = RFE(estimator=rf_model, n_features_to_select=5000, step=100)
X_resampled_rfe = rfe.fit_transform(X_resampled, y_resampled)

# Train the Final Random Forest Model
print("Training Final Random Forest Model...")
rf_model.fit(X_resampled_rfe, y_resampled)

# Calibrate Model with Isotonic Regression
print("Calibrating Model with Isotonic Regression...")
calibrated_rf = CalibratedClassifierCV(rf_model, method="isotonic", cv="prefit")
calibrated_rf.fit(X_resampled_rfe, y_resampled)

# Save the calibrated model
timestamp = time.strftime("%Y%m%d_%H%M%S")
model_filename = f"calibrated_rf_model_{timestamp}.pkl"
joblib.dump(calibrated_rf, model_filename)
print(f"Model saved as {model_filename}")
vectorizer_filename = f"tfidf_vectorizer_{timestamp}.pkl"
joblib.dump(tfidf_vectorizer, vectorizer_filename)
print(f"TF-IDF Vectorizer saved as {vectorizer_filename}.")
print("Model Training Complete!")


## Large Language Model Prompt (Llama 3.3)

In [ ]:
def evaluate_topic_with_llama(self, text: str) -> tuple[str, str]:
        try:
            prompt = f"""
            Analyze the following document sample and classify it into only one of these topics: {self.unique_topics_str}.
            After explaining your reasoning, clearly state the final topic and the confidence score at the end.
            
            Document:
            {text}
            
            Explanation: <Your explanation>
            Final Topic: <One of the topics from the list>
            Confidence Score: <0 to 100%>
            """
            formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """
            load_dotenv("codes.env")
            aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
            aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
            aws_region = os.environ.get("AWS_REGION")
            MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
            config = Config(read_timeout=1000)
            bedrock_client = boto3.client(
                "bedrock-runtime",
                region_name=aws_region,
                aws_access_key_id=aws_access_key,
                aws_secret_access_key=aws_secret_key,
                config=config
            )
            response = bedrock_client.invoke_model(
                modelId=MODEL_ID_LLAMA,
                body=json.dumps({
                    "prompt": formatted_prompt,
                    "max_gen_len": 512,
                    "temperature": 0,
                }),
                contentType="application/json"
            )
            response_body = json.loads(response['body'].read())
            response_text = response_body.get("generation", "").strip()
            match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
            predicted_topic = match.group(1).strip() if match else "Unknown"
            return predicted_topic, response_text
        except Exception as e:
            print(f"Error calling AWS Bedrock API: {e}")
            return "Unknown", "Error occurred during LLM call"

## Inference

In [ ]:
def classify_document(self) -> tuple[str, str, float, str, str]:
        """
        Classify a document using the pre-processed text.
        First, the Random Forest classifier is used; if its confidence is low, the fallback LLM is invoked.
        """
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)
        file_path = metadata["file_path"]
        filename = metadata["filename"]

        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        sampled_text = self.extract_intro_middle_conclusion(content)
        predicted_topic, confidence = self.rf_classify_document(sampled_text)

        if predicted_topic:
            print(f"Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
            return filename, predicted_topic, confidence, "-", "Random Forest Classification"

        predicted_topic, explanation = self.evaluate_topic_with_llama(sampled_text)
        print(f"LLM Classification: {predicted_topic}")
        return filename, predicted_topic, "-", explanation, "LLM Classification"

## Model Interpretability

In [ ]:
import lime
from lime.lime_text import LimeTextExplainer
def explain_prediction(self, text: str, method: str = "local") -> None:
        """
        Provide an explanation for a given prediction using one of several methods:
        
        - "local": A permutation-based local explanation that approximates feature importance.
        - "global": Uses the model's built-in global feature importances.
        - "shap": Computes and prints SHAP values for the input text (without visualization).
        
        Parameters:
            text (str): The document text to explain.
            method (str): The explanation method ("local", "global", "shap", or "lime").
        """
        import numpy as np
        # Convert the input text into a TF-IDF vector (dense format)
        text_tfidf = self.tfidf_vectorizer.transform([text])
        dense_vector = text_tfidf.toarray()  # shape: (1, n_features)
        feature_names = self.tfidf_vectorizer.get_feature_names_out()
        if len(feature_names) != dense_vector.shape[1]:
            feature_names = [f"f{i}" for i in range(dense_vector.shape[1])]
        
        # Get baseline prediction probability and predicted class.
        proba = self.rf_model.predict_proba(dense_vector)[0]
        predicted_class = self.rf_model.predict(dense_vector)[0]
        class_index = list(self.rf_model.classes_).index(predicted_class)
        baseline_prob = proba[class_index]
        
        if method == "local":
            # [Local explanation code unchanged...]
            candidate_indices = [i for i, val in enumerate(dense_vector[0]) if val > 0]
            candidate_indices = sorted(candidate_indices, key=lambda i: dense_vector[0][i], reverse=True)
            candidate_indices = candidate_indices[:20]
            
            feature_contributions = []
            for i in candidate_indices:
                modified_vector = dense_vector.copy()
                modified_vector[0][i] = 0.0  # Zero-out the feature's contribution
                new_prob = self.rf_model.predict_proba(modified_vector)[0][class_index]
                contribution = baseline_prob - new_prob
                feature_contributions.append((feature_names[i], contribution, dense_vector[0][i]))
            
            top_features = sorted(feature_contributions, key=lambda x: abs(x[1]), reverse=True)[:10]
            explanation_lines = [
                f"Predicted class: {predicted_class}",
                f"Baseline predicted probability: {baseline_prob:.4f}",
                "Top contributing features (feature: TF-IDF weight):"
            ]
            for feat, contrib, weight in top_features:
                explanation_lines.append(f"- {feat}: {weight:.4f}")
            print("\n".join(explanation_lines))
        
        elif method == "shap":
            # SHAP explanation: compute and print SHAP values per word.
            import shap
            # Compute SHAP values for the text
            explainer = shap.Explainer(self.rf_model.predict_proba, dense_vector, feature_names=feature_names)
            shap_values = explainer(dense_vector)
            # For multi-class, select SHAP values for the predicted class.
            shap_values_for_class = shap_values.values[0, :, class_index]
            
            # Only output words that are actually present (non-zero TF-IDF weight)
            nonzero_indices = [i for i, val in enumerate(dense_vector[0]) if val > 0]
            result_lines = [
                f"Word: {feature_names[i]}, TF-IDF: {dense_vector[0][i]:.4f}, SHAP: {shap_values_for_class[i]:.4f}"
                for i in nonzero_indices
            ]
            print("SHAP values for respective words:")
            print("\n".join(result_lines))
        
        elif method == "lime":
            explainer = LimeTextExplainer(class_names=self.rf_model.classes_)

            def predict_proba(texts):
                return self.rf_model.predict_proba(self.tfidf_vectorizer.transform(texts))

            exp = explainer.explain_instance(text, predict_proba, num_features=len(self.tfidf_vectorizer.get_feature_names_out()))
            print("Top words (features) influencing the prediction with their LIME scores:")
            sorted_features = sorted(exp.as_list(), key=lambda x: abs(x[1]), reverse=True)
            for feature, value in sorted_features[:5]:
                print(f"{feature}: {value}")
            return exp
        
        else:
            print(f"Explainability method '{method}' not implemented.")